# What is actually in this file

We will be using ~700 MB parquet file for this session. It is a DNA-encoded library screen
against **WDR91**, run by [HitGen](https://www.hitgen.com/en) and released through
[AIRCHECK](https://aircheck.ai/datasets). The modelling session starts after this one, and they
want a feature matrix from you.

Two blocks, about twelve minutes each. By the end you will have found two things about this
dataset that change what any model trained on it can possibly mean.

---

### How this works

Each block runs the same way:

1. **A worked example.** Complete, already run. Read it. *If it's familiar, skip straight to the exercise.*
2. **A prediction.** Commit a number before you compute it. You will often be wrong — that is the
   mechanism, not a failure. Nothing here is scored and nothing is compared to anyone else.
3. **An exercise**, with the last step left for you.
4. **A checkpoint cell** that sets everything up correctly whether or not you finished. You will
   never be stranded.

Every exercise has three tiers: 🟢 the core one, 🔵 a variation, and ⬛ something harder if you have
time. **Finishing 🟢 is finishing.**

`aircheck.hint("id")` for a nudge, `aircheck.reveal("id")` to just be told. Neither costs you anything.

## Setup

Everything is read from the shared data folder — nothing needs to be downloaded.


In [0]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")


def find_dir():
    """Locate the folder holding aircheck.py, wherever the notebook was launched from."""
    for candidate in [Path.cwd(), Path.cwd() / "challenge_v2", *Path.cwd().parents]:
        if (candidate / "aircheck.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find aircheck.py — run this notebook from the folder it came in.")


HERE = find_dir()
sys.path.insert(0, str(HERE))
import aircheck


def find_data(filename="train_wdr91_full.parquet"):
    candidates = [
        Path("/Volumes/uhn_workshop/lab/input/Train") / filename,  # Databricks volume
        *[root / sub / filename
          for root in [Path.cwd(), HERE, HERE.parent, HERE.parent.parent]
          for sub in ["sample_data", "."]],  # local fallbacks
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename}.")


DATA = find_data()
print(f"{DATA.name}  ·  {DATA.stat().st_size / 1e9:.2f} GB on disk")



---
# Block 1 · What `LABEL` actually means

**⏱ ~12 minutes**

Every model built after this session will be trained on the `LABEL` column. Nobody has told you
what it means. Before anyone fits anything, that is worth ten minutes.

## Worked example — load the metadata and see what is really there

The fingerprint columns are enormous; the metadata columns are not. Load those and audit them.

The move worth noticing is `.isna().mean()` rather than `.isna().sum()` — a **fraction** rather
than a count, because a fraction of exactly 1.0 is a very different problem from a large count.

In [0]:
meta_cols = ["COMPOUND_ID", "LIBRARY_ID", "BB1_ID", "BB2_ID", "BB3_ID", "TARGET_ID",
             "TARGET_VALUE", "NTC_VALUE", "ENRICHMENT", "LABEL", "SMILES", "MW", "ALOGP"]

meta = pq.read_table(DATA, columns=meta_cols).to_pandas()
print(f"{len(meta):,} rows x {meta.shape[1]} columns\n")

null_fraction = meta.isna().mean().sort_values(ascending=False)
print((null_fraction * 100).round(1).to_string())

Three columns are empty all the way down. `df.head()` would never have told you that — it shows
`None` in the first few rows of a mostly-full column too.

In [0]:
# fully_null = ["..."]     # <- names of the columns that are 100% empty
# fully_null = ["..."]     # <- names of the columns that are 100% empty
fully_null = sorted(null_fraction[null_fraction == 1.0].index)

aircheck.check("b1_null", fully_null)

### 💭 Concept check

`SMILES` being empty is the expensive one. Given that, and given the columns that *do* have data:

**`LABEL == 1` for a compound means:**

- **a)** its `ENRICHMENT` score is above 1
- **b)** its signal beat the no-target control, `NTC_VALUE`
- **c)** its raw sequencing count `TARGET_VALUE` reached some threshold
- **d)** it was measured as a potent binder in a follow-up assay

In [0]:
# aircheck.mcq("b1_label", ...)      # "a", "b", "c" or "d"
aircheck.mcq("b1_label", "c")      # "a", "b", "c" or "d"

## 🔮 Predict

So `LABEL` is a threshold on `TARGET_VALUE`. You are about to work out where that threshold sits,
and then count how many compounds landed *just below* it — the near-misses.

**Out of 375,595 compounds, how many were sequenced between 1 and 5 times?**

Commit a number. Being wrong is the point.

In [0]:
# aircheck.predict("b1_middle", ...)
aircheck.predict("b1_middle", 40000)

## 🟢 Exercise — find the threshold, then count the near-misses

`groupby` on the label and describe `TARGET_VALUE` on each side. The threshold falls straight out
of the table.

In [0]:
by_label = meta.groupby("LABEL")["TARGET_VALUE"].agg(["min", "max", "median", "count"])
display(by_label)
print()

# threshold = ...      # <- read it straight off the table above
threshold = int(meta.loc[meta.LABEL == 1, "TARGET_VALUE"].min())


aircheck.check("b1_threshold", threshold)

In [0]:
counts = meta["TARGET_VALUE"]

# n_middle = ...       # <- compounds sequenced between 1 and threshold-1 times
n_middle = int(((counts > 0) & (counts < threshold)).sum())

print(f"count == 0          {int((counts == 0).sum()):>8,}")
print(f"count 1..{threshold - 1}            {n_middle:>8,}")
print(f"count >= {threshold}           {int((counts >= threshold).sum()):>8,}")
print()

aircheck.check("b1_middle", n_middle)

In [0]:
# 🧭 Checkpoint — runs correctly whether or not you finished above.
threshold = int(meta.loc[meta.LABEL == 1, "TARGET_VALUE"].min())
meta["LIB"] = meta["LIBRARY_ID"].str.split("-").str[0]
aircheck.checkpoint()

### 🔵 If you have a minute

`TARGET_VALUE` for the hits is steeply skewed — the median hit has 9 reads and the largest has 696.
Change the cutoff below and watch how many "hits" survive. There is nothing to submit; the point is
to feel how arbitrary a single threshold is.

In [0]:
def hits_at(cutoff):
    """How many compounds would be hits if the threshold had been set here instead?"""
    n = int((meta.TARGET_VALUE >= cutoff).sum())
    print(f"cutoff {cutoff:>4}  ->  {n:>7,} hits  ({n / len(meta):.2%} of the entire dataset)")


for cutoff in [6, 10, 20, 50, 100]:
    hits_at(cutoff)

# try your own:
hits_at(6)

---
# Block 2 · It is 39 libraries, not one dataset

**⏱ ~12 minutes**

`LIBRARY_ID` looks like an opaque key: `L23-0109-0043-0510`. It is not opaque — it is four fields
glued together. The first is the library; the other three are the building blocks combined to make
the compound.

That first field turns out to be the most important column in the file, and nothing in the schema
says so.

## Worked example — split it out and look at the libraries separately

In [0]:
# meta["LIB"] was set in the checkpoint above.
library_summary = meta.groupby("LIB").agg(
    n_compounds=("LABEL", "size"),
    n_hits=("LABEL", "sum"),
    hit_rate=("LABEL", "mean"),
).sort_values("hit_rate", ascending=False)

# A library with 2 compounds and 1 hit is not a 50% hit rate, it is a rounding error.
big = library_summary[library_summary.n_compounds >= 1000]

print(f"{meta.LIB.nunique()} distinct libraries, {len(big)} with at least 1,000 compounds\n")
display(big.head(5).style.format({"hit_rate": "{:.1%}"}))

In [0]:
fig, ax = plt.subplots(figsize=(11, 4))
order = big.sort_values("hit_rate")
ax.bar(order.index, order.hit_rate, color="#4c72b0")
ax.axhline(meta.LABEL.mean(), color="#c44e52", ls="--",
           label=f"whole file: {meta.LABEL.mean():.1%}")
ax.set(ylabel="hit rate", xlabel="library",
       title="Same protein, same assay, different libraries")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## 🔮 Predict

**By what factor do the best and worst libraries differ in hit rate?** (1 means they are all the
same; 10 means an order of magnitude apart.)

In [0]:
# aircheck.predict("b2_spread", ...) # placeholder guess, not the answer -- the aircheck.check cell below is what is graded
aircheck.predict("b2_spread", 2)

## 🟢 Exercise — measure the spread

In [0]:
spread = ...     # <- highest hit rate over lowest, among libraries with >= 1000 compounds
spread = big.hit_rate.max() / big.hit_rate.min()


print(f"best:  {big.hit_rate.idxmax()}  {big.hit_rate.max():.1%}")
print(f"worst: {big.hit_rate.idxmin()}  {big.hit_rate.min():.1%}")
print()

aircheck.check("b2_spread", spread)

## The same fact, wearing a different hat

Here is a claim you will find in a lot of DEL tutorials, including an earlier version of this one:

> *"Molecular weight and logP don't separate hits from non-hits — the distributions overlap almost
> completely."*

Measured across the whole file, that is true. AUC is a clean way to say it: **0.5 means no
separation at all**, and above 0.5 means hits tend to be heavier.

In [0]:
def mw_auc(frame):
    """P(a random hit is heavier than a random non-hit). 0.5 = no separation."""
    hits, misses = frame[frame.LABEL == 1].MW, frame[frame.LABEL == 0].MW
    if len(hits) < 30 or len(misses) < 30:
        return np.nan
    return stats.mannwhitneyu(hits, misses).statistic / (len(hits) * len(misses))


print(f"whole file: MW AUC = {mw_auc(meta):.3f}   <- looks like no signal")

## 🟢 Exercise — now ask the same question one library at a time

Compute the MW AUC **inside library L37**, then inside **L04**, and compare them to the number above.

In [0]:
# auc_l37 = ...      # <- MW AUC using only L37 compounds
# auc_l04 = ...      # <- and only L04 compounds

auc_l37 = mw_auc(meta[meta.LIB == "L37"])
auc_l04 = mw_auc(meta[meta.LIB == "L04"])

print(f"L37: {auc_l37:.3f}")
print(f"L04: {auc_l04:.3f}")
print()

aircheck.check("b2_mw_auc", auc_l37)

In [0]:
# 🧭 Checkpoint
auc_l37, auc_l04 = mw_auc(meta[meta.LIB == "L37"]), mw_auc(meta[meta.LIB == "L04"])

per_library = (meta.groupby("LIB").filter(lambda g: len(g) >= 2000)
                   .groupby("LIB").apply(mw_auc, include_groups=False)
                   .dropna().sort_values())

fig, ax = plt.subplots(figsize=(10, 4))
colours = ["#c44e52" if v < 0.5 else "#4c72b0" for v in per_library]
ax.bar(per_library.index, per_library.values, color=colours)
ax.axhline(0.5, color="#333", ls="--", label="no separation")
ax.axhline(mw_auc(meta), color="#55a868", ls=":", lw=2,
           label=f"pooled across all libraries: {mw_auc(meta):.3f}")
ax.set(ylim=(0.35, 0.8), ylabel="MW AUC", xlabel="library",
       title="Molecular weight, asked one library at a time")
ax.legend()
plt.tight_layout()
plt.show()

aircheck.checkpoint("Red bars run the opposite way to blue ones. That is why the pooled number is ~0.5.")

## The third time, in a picture

The last check is chemical space itself: **do compounds cluster by what they bind, or by where
they were made?** To see that you need 4,000 molecules, each described by 2,048 fingerprint bits,
drawn on a flat page. That is what UMAP is for.

### What UMAP does

**UMAP** (Uniform Manifold Approximation and Projection) draws high-dimensional points in two
dimensions so that *neighbours stay neighbours*. It works in two steps:

1. **Find each compound's nearest neighbours in fingerprint space.** With `n_neighbors=20`, every
   molecule is linked to the 20 it most resembles. "Resembles" is measured with **Jaccard distance**
   (1 minus Tanimoto similarity): of the bits set in either fingerprint, what share is set in both?
   Euclidean distance would ask the wrong question — on sparse binary fingerprints it mostly counts
   how many bits are set, which is molecule size.
2. **Lay the points out in 2-D so that linked molecules land close together.** The layout is found
   by stochastic gradient descent, which is why it takes half a minute and why `random_state=42` is
   set: a different seed gives a different picture of the same structure. `min_dist=0.1` sets how
   tightly points may pack; lower gives denser islands.

The result is a **neighbourhood map, not a projection.** Local structure is roughly preserved;
global structure is not. Two islands far apart are not more different than two side by side, the
axes have no units, and an island's size on the page says nothing about how many molecules it holds.

### What HDBSCAN adds

A picture is not a number. **HDBSCAN** is a density-based clustering that looks for islands in the
2-D layout — regions packed more tightly than their surroundings — and gives each one a cluster ID.
Points in sparse regions get `-1` and are left out. `min_cluster_size=25` means an island needs at
least 25 compounds to count. With clusters attached, the exercise below can ask a precise question
of each island instead of squinting at colours.

### What goes in

One batch of 4,000 rows straight off the disk with `iter_batches` — not the whole 650 MB file. The
`ECFP4` column holds 2,048 counts per compound; they are turned into 0/1 (does the substructure
occur, not how often) because Jaccard is defined on sets.

UMAP takes about 30 s on 4,000 × 2,048. Run the cell, then read the above while it works. If your
cluster count differs slightly from your neighbour's, that is UMAP being UMAP; the check allows for it.

In [0]:
import time
import warnings

# Three warnings raised on the way in are noise for us: tqdm wanting ipywidgets for a progress
# bar, no inverse_transform for Jaccard (we never go from the picture back to a fingerprint),
# and a fixed seed disabling parallelism (the price of a reproducible picture, paid on purpose).
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="gradient function is not yet implemented")
warnings.filterwarnings("ignore", message="n_jobs value .* overridden")

import umap
import hdbscan

N_SAMPLE = 4000
UMAP_KWARGS = dict(n_components=2, n_neighbors=20, min_dist=0.1,
                   metric="jaccard", random_state=42)

# One batch off the disk, not the whole file.
batch = next(pq.ParquetFile(DATA).iter_batches(
    batch_size=N_SAMPLE, columns=["ECFP4", "LABEL", "LIBRARY_ID"]))
sample = pa.Table.from_batches([batch])

# ECFP4 is a list column: 2,048 counts per compound. Flatten to a matrix, then binarise —
# Jaccard is a question about sets, so "does the substructure occur" rather than "how often".
column = sample.column("ECFP4").combine_chunks()
counts = column.flatten().to_numpy(zero_copy_only=False).reshape(len(column), -1)
binary = (counts > 0).astype(np.float32)
print(f"{binary.shape[0]:,} compounds x {binary.shape[1]:,} bits, "
      f"{binary.mean():.1%} of bits set on average")

t0 = time.perf_counter()
embedding = umap.UMAP(**UMAP_KWARGS).fit_transform(binary)
print(f"UMAP: {time.perf_counter() - t0:.0f} s")

clusters = hdbscan.HDBSCAN(min_cluster_size=25).fit_predict(embedding)

space = sample.select(["LABEL", "LIBRARY_ID"]).to_pandas()
space["LIB"] = space["LIBRARY_ID"].str.split("-").str[0]
space["UMAP1"], space["UMAP2"] = embedding[:, 0], embedding[:, 1]
space["CLUSTER"] = clusters

n_clusters = int((np.unique(clusters) >= 0).sum())
print(f"{space.LIB.nunique()} libraries in the sample  ·  {n_clusters} clusters  ·  "
      f"{(clusters < 0).mean():.0%} of compounds in no cluster")

Look at the left panel, then the right one. Same 4,000 points, same layout — only the colours change.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].scatter(space.UMAP1, space.UMAP2, s=6, c="#b9c0cc", alpha=0.5)
hits = space[space.LABEL == 1]
axes[0].scatter(hits.UMAP1, hits.UMAP2, s=16, c="#d62728", alpha=0.85)
axes[0].set_title("coloured by activity")

top_libs = space.LIB.value_counts().head(10).index
for lib, colour in zip(top_libs, plt.cm.tab10.colors):
    part = space[space.LIB == lib]
    axes[1].scatter(part.UMAP1, part.UMAP2, s=6, alpha=0.65, color=colour, label=lib)
rest = space[~space.LIB.isin(top_libs)]
axes[1].scatter(rest.UMAP1, rest.UMAP2, s=4, c="#dddddd", alpha=0.4)
axes[1].legend(markerscale=3, fontsize=8, ncol=2, loc="upper left")
axes[1].set_title("coloured by library")

for ax in axes:
    ax.set(xlabel="UMAP1", ylabel="UMAP2")
plt.tight_layout()
plt.show()

## 🟢 Exercise — put a number on it


For each cluster, calculate the percentage of compounds that come from the same library. A value of 1.0 means all compounds come from one library.

In [0]:
clustered = space[space.CLUSTER >= 0]

# purity = ...     # <- one number per cluster: the share of its most common library
purity = clustered.groupby("CLUSTER")["LIB"].apply(
    lambda s: s.value_counts().iloc[0] / len(s))

summary = clustered.groupby("CLUSTER").agg(n=("LABEL", "size"), hit_rate=("LABEL", "mean"))
summary["library_purity"] = purity
summary["top_library"] = clustered.groupby("CLUSTER")["LIB"].agg(lambda s: s.value_counts().index[0])
display(summary.sort_values("n", ascending=False).head(6).round(3))
print()

aircheck.check("b2_purity", purity.mean())

### 💭 Concept check

You have now found the same fact three times: hit rate varies 16-fold by library, molecular weight
trends reverse between libraries, and chemical-space clusters are mostly single libraries.

**The modelling session is about to split this data into train and test. What should they do?**

- **a)** Split at random — 375,595 rows is plenty, so the split will be representative
- **b)** Group the folds by library, so a library in training is absent from test
- **c)** Drop the `LIBRARY_ID` column so the model cannot use it
- **d)** Resample so every library has the same hit rate

In [0]:
# aircheck.mcq("b2_confound", ...)
aircheck.mcq("b2_confound", "b")

### ⬛ If you got here early

`mw_auc` above only looks at molecular weight. Write the ALOGP version and run it per library —
the reversal is even stronger there (it ranges from about 0.42 to 0.72). Then ask yourself which
*single* library you would hand a modeller if they only had time to fit one.

In [0]:
# your turn
def alogp_auc(frame):
    hits, misses = frame[frame.LABEL == 1].ALOGP, frame[frame.LABEL == 0].ALOGP
    if len(hits) < 30 or len(misses) < 30:
        return np.nan
    return stats.mannwhitneyu(hits, misses).statistic / (len(hits) * len(misses))


per_lib_alogp = (meta.groupby("LIB").filter(lambda g: len(g) >= 2000)
                     .groupby("LIB").apply(alogp_auc, include_groups=False).dropna())
print(f"ALOGP AUC by library: {per_lib_alogp.min():.3f} to {per_lib_alogp.max():.3f}")
print(f"pooled across everything: {alogp_auc(meta):.3f}")

---
## Where you got to

In [0]:
aircheck.progress()

### What to carry into the next notebook

Two things, and they are the whole reason this block existed:

1. **`LABEL = 0` does not mean inactive.** It means sequenced zero times, and every ambiguous
   compound was deleted before you saw the file.
2. **This is 39 libraries pooled, not one dataset.** Hit rate, physicochemical trend and chemical
   space are all distorted by that single fact — so a model can score well here while learning
   nothing at all about WDR91.

Neither of those needed a model. Both change what a model would mean.

➡️ **Next: `2_Build_features.ipynb`** — where you build the matrix you hand over.